In [10]:
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
from speechbrain.inference import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa-voxceleb",
    local_strategy=LocalStrategy.COPY
)

c:\Users\Mozalan\Desktop\voice_recognition\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mozalan\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
signal = classifier.load_audio("../data/processed/train/Sungho_0.wav")
embedding = classifier.encode_batch(signal)
print(embedding.shape)

torch.Size([1, 1, 192])


In [4]:
def extract_embedding(file_path):
    signal = classifier.load_audio(file_path)
    embedding = classifier.encode_batch(signal)
    embedding = embedding.squeeze().numpy()
    return embedding

In [6]:
import os
import numpy as np

train_folder = "../data/processed/train/"
train_files = [f for f in os.listdir(train_folder) if f.endswith(".wav")]

X_train_emb_list = []
y_train_emb_list = []

for filename in train_files:
    member_name = filename.split("_")[0]
    y_train_emb_list.append(member_name)
    
    file_path = train_folder + filename
    emb = extract_embedding(file_path)
    X_train_emb_list.append(emb)

X_train_emb = np.array(X_train_emb_list)
y_train_emb = np.array(y_train_emb_list)

print(X_train_emb.shape)
print(y_train_emb.shape)

(48, 192)
(48,)


test file

In [7]:
test_folder = "../data/processed/test/"
test_files = [f for f in os.listdir(test_folder) if f.endswith(".wav")]

X_test_emb_list = []
y_test_emb_list = []

for filename in test_files:
    member_name = filename.split("_")[0]
    y_test_emb_list.append(member_name)
    
    file_path = test_folder + filename
    emb = extract_embedding(file_path)
    X_test_emb_list.append(emb)

X_test_emb = np.array(X_test_emb_list)
y_test_emb = np.array(y_test_emb_list)

print(X_test_emb.shape)
print(y_test_emb.shape)

(14, 192)
(14,)


In [8]:
member_list_emb = sorted(set(y_train_emb))
centroids = []

for member in member_list_emb:
    indices = [i for i, label in enumerate(y_train_emb) if label == member]
    centroid = X_train_emb[indices].mean(axis=0)
    centroids.append(centroid)

centroids = np.array(centroids)
print(centroids.shape)

(6, 192)


In [11]:
from scipy.spatial.distance import cdist

distances = cdist(X_test_emb, centroids, metric='euclidean')
predicted_indices = np.argmin(distances, axis=1)
y_pred_centroid = np.array([member_list_emb[i] for i in predicted_indices])

print("Accuracy:", accuracy_score(y_test_emb, y_pred_centroid))
print(confusion_matrix(y_test_emb, y_pred_centroid, labels=member_list_emb))

Accuracy: 0.7857142857142857
[[2 0 0 0 0 1]
 [0 2 0 0 0 0]
 [0 0 2 0 0 0]
 [0 0 0 2 0 0]
 [0 0 0 1 2 0]
 [1 0 0 0 0 1]]
